## Preliminaries

### Import statements

In [1]:
import os
import csv
import re
import git
import bisect
import platform, psutil, torch
from time import time
from collections import Counter
from copy import deepcopy
import requests
from lxml import etree
import pandas as pd
import spacy
from MyCapytain.resolvers.cts.local import CtsCapitainsLocalResolver
from MyCapytain.resources.prototypes.metadata import UnknownCollection
from dicesapi import DicesAPI

### Global values

In [2]:
# local data directory, used for both input and output
data_dir = "data"

# corpus repositories
repositories = [
    {
        "label": "Perseus Greek",
        "base_url": "https://github.com/perseusDL",
        "repo_name": "canonical-greekLit",
        "commit": "beed7ea8926266ad90935c183f2bcf2caf4ac0dc",
    },
]
# XML namespaces used by Perseus TEI, needed for xpath
nsmap = {
    "cts": "http://chs.harvard.edu/xmlns/cts",
    "tei": "http://www.tei-c.org/ns/1.0",
    "py": "http://codespeak.net/lxml/objectify/pytype",
}

# https://huggingface.co/chcaa/grc_odycy_joint_trf/blob/main/grc_odycy_joint_trf-0.7.0-py3-none-any.whl
spacy_model = "grc_odycy_joint_trf"

## Get texts

### Create local clone of Perseus repository

In [3]:
print('Checking for local text repositories...')
for rec in repositories:
    dest_path = os.path.join(data_dir, rec["repo_name"])
    remote_url = rec["base_url"] + "/" + rec["repo_name"] + ".git"

    if os.path.exists(dest_path):
        print(f' - {dest_path} exists!')
        repo = git.Repo(dest_path)
    else:
        print(f' - retrieving {dest_path}')
        repo = git.Repo.clone_from(remote_url, dest_path)

    # reset to specified commit
    print(f" - resetting to commit {rec['commit']}")
    repo.head.reset(rec["commit"], working_tree=True)
        

Checking for local text repositories...
 - data/canonical-greekLit exists!
 - resetting to commit beed7ea8926266ad90935c183f2bcf2caf4ac0dc


### Read text metadata

In [4]:
corpus_file = os.path.join(data_dir, "corpus.tsv")
print(f"Reading text metadata from {corpus_file}")

corpus = []

with open(corpus_file) as f:
    reader = csv.DictReader(f, delimiter="\t")
    for rec in reader:
        corpus.append(dict(
            author = rec["author"],
            title = rec["title"],
            urn = rec["urn"],
            prefix = rec["prefix"],
        ))
print(f" - got data on {len(corpus)} texts")

Reading text metadata from data/corpus.tsv
 - got data on 115 texts


### Load XML using CTS API

#### initialize local resolver

In [5]:
repo_paths = [os.path.join(data_dir, rec["repo_name"]) for rec in repositories]
resolver = CtsCapitainsLocalResolver(repo_paths)

#### retrieve the texts

In [6]:
print("Retrieving texts...")
for text in corpus:
    try:
        text["xml"] = resolver.getTextualNode(text["urn"], text["prefix"]).xml
    except UnknownCollection:
        print(f" - {text['author']} {text['title']} {text['prefix']} failed")

print(len([text for text in corpus if text.get("xml") is not None]), "texts retrieved")

Retrieving texts...
115 texts retrieved


### Extract verse lines

#### build an array of lines

In [7]:
print("Building line arrays...")
nlines = 0

for i, text in enumerate(corpus):
    print(f"[{i+1}/{len(corpus)}]", text["author"], text["title"], text["prefix"], end="\t")

    # bail if no XML
    if text.get("xml") is None:
        print("failed: no XML")
        continue

    # initialize line array
    text['line_array'] = []

    # work from copy of xml
    xml = deepcopy(text["xml"])
        
    # remove notes
    for note in xml.findall(".//tei:note", namespaces=nsmap):
        note.clear(keep_tail=True)
        
    # remove deleted lines
    for del_ in xml.findall(".//tei:del", namespaces=nsmap):
        del_.clear(keep_tail=True)
        
    # iterate over verse lines
    for l in xml.findall(".//tei:l", namespaces=nsmap):
        line_num = l.get("n")
        if line_num is None:
            continue
        
        line_text = "".join(s for s in l.itertext())
        line_text = re.sub(r"\s+", " ", line_text).strip()
        text['line_array'].append(dict(
            n = line_num,
            seq = len(text['line_array']),
            text = line_text,
        ))

    print(len(text["line_array"]), "lines")
    nlines += len(text["line_array"])
print(f"Total: {nlines} lines")

Building line arrays...
[1/115] Homer Iliad 1	611 lines
[2/115] Homer Iliad 2	877 lines
[3/115] Homer Iliad 3	461 lines
[4/115] Homer Iliad 4	544 lines
[5/115] Homer Iliad 5	909 lines
[6/115] Homer Iliad 6	529 lines
[7/115] Homer Iliad 7	482 lines
[8/115] Homer Iliad 8	565 lines
[9/115] Homer Iliad 9	709 lines
[10/115] Homer Iliad 10	579 lines
[11/115] Homer Iliad 11	847 lines
[12/115] Homer Iliad 12	471 lines
[13/115] Homer Iliad 13	837 lines
[14/115] Homer Iliad 14	521 lines
[15/115] Homer Iliad 15	746 lines
[16/115] Homer Iliad 16	867 lines
[17/115] Homer Iliad 17	761 lines
[18/115] Homer Iliad 18	617 lines
[19/115] Homer Iliad 19	424 lines
[20/115] Homer Iliad 20	503 lines
[21/115] Homer Iliad 21	611 lines
[22/115] Homer Iliad 22	515 lines
[23/115] Homer Iliad 23	897 lines
[24/115] Homer Iliad 24	804 lines
[25/115] Homer Odyssey 1	444 lines
[26/115] Homer Odyssey 2	434 lines
[27/115] Homer Odyssey 3	497 lines
[28/115] Homer Odyssey 4	847 lines
[29/115] Homer Odyssey 5	493 lines
[30

#### index lines by starting character position

In [8]:
print("Building line indices...")
success = 0

for text in corpus:
    # start index at zero
    text["line_index"] = []
    cumsum = 0

    # iterate over line array, add length (plus 1 for spaces between lines)
    for line in text["line_array"]:
        text["line_index"].append(cumsum)
        cumsum += len(line["text"]) + 1

    # make sure the count works out
    if cumsum != len(" ".join(line["text"] for line in text["line_array"])) + 1:
        print(" - character count doesn't match:", text["author"], text["title"], text["prefix"])
    else:
        success += 1

print(success, " indices built successfully")

Building line indices...
115  indices built successfully


## Parse text with spaCy
### load spacy model

In [9]:
nlp = spacy.load(spacy_model)

### parse each text as one long string

Times below reflect the testing system:
- MacBook Pro M3
- 12 cores
- 18 GB RAM
- MPS acceleration

In [10]:
print(f"Parsing with {spacy_model}...")
_t0 = time()

for i, text in enumerate(corpus):
    print(f"[{i+1}/{len(corpus)}]", text["author"], text["title"], text["prefix"], end="\t")
    t0 = time()

    one_long_string = " ".join(line["text"] for line in text["line_array"])
    text["spacy_doc"] = nlp(one_long_string)

    t1 = time()
    print(len(text["spacy_doc"]), "tokens,", int(t1-t0), "seconds")

# format total run time
total_time = time()-_t0
h = int(total_time / 3600)
m = int((total_time - h*3600) / 60)
s = int(total_time - h*3600 - m*60)
print(f"total time: {h}:{m:02d}:{s:02d}")

Parsing with grc_odycy_joint_trf...
[1/115] Homer Iliad 1	

/Users/chris/Documents/git/uva-common/venv/lib/python3.10/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


5140 tokens, 9 seconds
[2/115] Homer Iliad 2	6822 tokens, 12 seconds
[3/115] Homer Iliad 3	3686 tokens, 6 seconds
[4/115] Homer Iliad 4	4341 tokens, 8 seconds
[5/115] Homer Iliad 5	7306 tokens, 13 seconds
[6/115] Homer Iliad 6	4247 tokens, 7 seconds
[7/115] Homer Iliad 7	3867 tokens, 7 seconds
[8/115] Homer Iliad 8	4560 tokens, 9 seconds
[9/115] Homer Iliad 9	5778 tokens, 10 seconds
[10/115] Homer Iliad 10	4751 tokens, 8 seconds
[11/115] Homer Iliad 11	6860 tokens, 12 seconds
[12/115] Homer Iliad 12	3741 tokens, 6 seconds
[13/115] Homer Iliad 13	6684 tokens, 12 seconds
[14/115] Homer Iliad 14	4241 tokens, 7 seconds
[15/115] Homer Iliad 15	6063 tokens, 11 seconds
[16/115] Homer Iliad 16	6970 tokens, 13 seconds
[17/115] Homer Iliad 17	6134 tokens, 11 seconds
[18/115] Homer Iliad 18	5030 tokens, 9 seconds
[19/115] Homer Iliad 19	3460 tokens, 6 seconds
[20/115] Homer Iliad 20	4080 tokens, 7 seconds
[21/115] Homer Iliad 21	5074 tokens, 9 seconds
[22/115] Homer Iliad 22	4326 tokens, 7 second

### Create token table

In [11]:
tokens = []

for text in corpus:
    for token in text["spacy_doc"]:
        i = bisect.bisect_right(text["line_index"], token.idx) - 1
        locus = text["line_array"][i]["n"]
        if locus is None:
            continue
            
        if text["prefix"]:
            locus = text["prefix"] + "." + locus

        tokens.append(dict(
            urn = text["urn"] + ":" + locus,
            author = text["author"],
            work = text["title"],
            pref = text["prefix"],
            seq = text["line_array"][i]["seq"],
            line = text["line_array"][i]["n"],
            token = token.text,
            lemma = token.lemma_,
            pos = token.pos_,
            verbform = ";".join(token.morph.get("VerbForm")),                
            mood = ";".join(token.morph.get("Mood")),
            tense = ";".join(token.morph.get("Tense")),                
            voice = ";".join(token.morph.get("Voice")),
            person = ";".join(token.morph.get("Person")),
            number = ";".join(token.morph.get("Number")),
            case = ";".join(token.morph.get("Case")),
            gender = ";".join(token.morph.get("Gender")),
        ))

tokens = pd.DataFrame(tokens)

## Identify speech/narration

### instantiate API

In [12]:
api = DicesAPI(logfile="dices.log")

### download full speech list

In [13]:
all_speeches = api.getSpeeches()
print(len(all_speeches), "speeches")

4689 speeches


### edit two speeches

This is necessary to fix line differences between editions used by DICES and Perseus

In [14]:
for s in all_speeches:
    # missing line number in Apollonius
    if s.work.urn=="urn:cts:greekLit:tlg0001.tlg001.perseus-grc2" and s.l_la=="3.739":
        s.l_la = "3.738"

    # missing line number in Odyssey
    if s.work.urn=="urn:cts:greekLit:tlg0012.tlg002.perseus-grc2" and s.l_fi=="10.456":
        s.l_fi = "10.455"

### create a table of all line numbers in order

In [15]:
all_lines = []
for text in corpus:
    for line in text["line_array"]:
        locus = line["n"]
        if locus is None:
            continue
        if text["prefix"]:
            locus = text["prefix"] + "." + locus
        all_lines.append(dict(
            urn = text["urn"] + ":" + locus,
            speech_id = None,
            speaker = None,
            addressee = None,
            level = 0,
        ))
all_lines = pd.DataFrame(all_lines)

### mark all lines that belong to speeches

In [16]:
urns = [text["urn"] for text in corpus]

for speech in all_speeches:
    
    # skip speeches that don't appear in our corpus
    if speech.work.urn not in urns:
        continue

    # determine first and last rows in table
    i_first = all_lines.index[
        all_lines["urn"].eq(speech.work.urn + ":" + speech.l_fi)
        ][0]
    i_last = all_lines.index[
        all_lines["urn"].eq(speech.work.urn + ":" + speech.l_la)
        ][0]
    
    # set speech attributes for all rows in between
    all_lines.loc[i_first:i_last, "speech_id"] = speech._attributes["public_id"]
    all_lines.loc[i_first:i_last, "speaker"] = speech.getSpkrString()
    all_lines.loc[i_first:i_last, "addressee"] = speech.getAddrString()
    all_lines.loc[i_first:i_last, "level"] = speech.level + 1
    

### mark Odysseus' apologue as different from his other speech

In [17]:
apologue_speech_ids = ["3F63", "A2AB"]

all_lines.loc[all_lines["speech_id"].isin(apologue_speech_ids), "speaker"] = "Odysseus-Apologue"

### Merge line table with token table to link tokens with speeches

In [18]:
df = pd.merge(all_lines, tokens, on="urn")

## Export token table

In [19]:
output_file = os.path.join(data_dir, "tokens.tsv")
df.to_csv(output_file, index=False, sep="\t")
print(f"{len(df)} tokens exported to {output_file}")

496427 tokens exported to data/tokens.tsv


## Sanity check
### how many tokens of narrative, speech do we have?

In [20]:
pd.crosstab(df["level"], df["work"])

work,Argonautica,Dionysiaca,Iliad,Odyssey,Posthomerica,Sack of Troy
level,,,,,,
0,31560,92115,69319,32030,51632,3749
1,13747,55169,57432,60793,16679,1058
2,0,1086,566,9468,0,0
3,0,24,0,0,0,0


### are any lines being counted twice?

In [21]:
df.groupby(["urn"]).agg(count = ("speech_id", "nunique")).query("count>1")

,count
urn,
